Libraries

In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

Import Data From sklearn

In [2]:
mnist_df = fetch_openml('mnist_784', version=1, as_frame=False)
X = mnist_df.data.astype(np.float32) / 255.0
y = mnist_df.target.astype(np.int64)

Split Data

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=22, stratify=y)

In [4]:
X_train = X_train.T
X_test = X_test.T



Encoding Labels

In [5]:
encoder = OneHotEncoder(sparse_output=False)
y_train = encoder.fit_transform(y_train.reshape(-1,1)).T 
y_test = encoder.transform(y_test.reshape(-1,1)).T    

In [6]:
print(X_train.shape , y_train.shape, X_test.shape, y_test.shape)

(784, 49000) (10, 49000) (784, 21000) (10, 21000)


In [7]:
print(y_train)

[[0. 0. 0. ... 0. 1. 0.]
 [0. 1. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 1. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [8]:
import pandas as pd

df = pd.DataFrame(X_train.T)
df = pd.concat([df, pd.DataFrame(y_train.T, columns=[f'Label_{i}' for i in range(10)])], axis=1)
print(df.head())


     0    1    2    3    4    5    6    7    8    9  ...  Label_0  Label_1  \
0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...      0.0      0.0   
1  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...      0.0      1.0   
2  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...      0.0      0.0   
3  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...      1.0      0.0   
4  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...      1.0      0.0   

   Label_2  Label_3  Label_4  Label_5  Label_6  Label_7  Label_8  Label_9  
0      0.0      0.0      1.0      0.0      0.0      0.0      0.0      0.0  
1      0.0      0.0      0.0      0.0      0.0      0.0      0.0      0.0  
2      0.0      0.0      0.0      0.0      0.0      1.0      0.0      0.0  
3      0.0      0.0      0.0      0.0      0.0      0.0      0.0      0.0  
4      0.0      0.0      0.0      0.0      0.0      0.0      0.0      0.0  

[5 rows x 794 columns]


Layers and Activations

In [9]:
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01 * np.random.randn(n_neurons, n_inputs)
        self.biases = np.zeros((n_neurons, 1))
    
    def forward(self, inputs):
        self.inputs = inputs  
        self.output = np.dot(self.weights, inputs) + self.biases
    
    def backward(self, doutput, learning_rate):
        m = self.inputs.shape[1]
        dW = np.dot(doutput, self.inputs.T) / m
        db = np.sum(doutput, axis=1, keepdims=True) / m
        dinputs = np.dot(self.weights.T, doutput)
    
        self.weights -= learning_rate * dW
        self.biases -= learning_rate * db
        return dinputs

class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)
    
    def backward(self, doutput):
        dinputs = doutput.copy()
        dinputs[self.inputs <= 0] = 0
        return dinputs

class Activation_Softmax:
    def forward(self,inputs):
        exp_values = np.exp(inputs - np.max(inputs, axis=0, keepdims=True))
        probabilities = exp_values / np.sum(exp_values , axis=0,keepdims=True)
        self.output = probabilities
    
    def backward(self, Y):
        return self.output - Y

Loss function anc accuracy

In [10]:
def cross_entropy_loss(Ypred, Y):
    loss = -np.sum(Y * np.log(Ypred + 1e-15)) / Y.shape[1]
    return loss

def accuracy(Ypred, Y):
    return np.mean(np.argmax(Ypred, axis=0) == np.argmax(Y, axis=0))

objects

In [11]:
dense1 = Layer_Dense(n_inputs=784, n_neurons=128)
dense2 = Layer_Dense(n_inputs=128, n_neurons=10)
ReLU = Activation_ReLU()
softmax = Activation_Softmax()

Learning

In [12]:
iterations = 100
learning_rate = 0.1
for iteration in range(iterations):

    dense1.forward(X_train)
    ReLU.forward(dense1.output)
    dense2.forward(ReLU.output)
    softmax.forward(dense2.output)

    loss = cross_entropy_loss(softmax.output , y_train)
    acc = accuracy(softmax.output,y_train)

    dsoftmax = softmax.backward(y_train)
    d_dense2 = dense2.backward(dsoftmax ,learning_rate)
    d_relu = ReLU.backward(d_dense2)
    dense1.backward(d_relu,learning_rate)

    print(f"iteration {iteration+1}/{iterations} - Loss: {loss:.4f} - Accuracy: {acc:.4f}")
    learning_rate = 0.2 * loss 



iteration 1/100 - Loss: 2.3022 - Accuracy: 0.1211
iteration 2/100 - Loss: 2.3010 - Accuracy: 0.1616
iteration 3/100 - Loss: 2.2954 - Accuracy: 0.3585
iteration 4/100 - Loss: 2.2889 - Accuracy: 0.5328
iteration 5/100 - Loss: 2.2806 - Accuracy: 0.6105
iteration 6/100 - Loss: 2.2693 - Accuracy: 0.6430
iteration 7/100 - Loss: 2.2539 - Accuracy: 0.6562
iteration 8/100 - Loss: 2.2328 - Accuracy: 0.6594
iteration 9/100 - Loss: 2.2041 - Accuracy: 0.6568
iteration 10/100 - Loss: 2.1659 - Accuracy: 0.6537
iteration 11/100 - Loss: 2.1162 - Accuracy: 0.6535
iteration 12/100 - Loss: 2.0535 - Accuracy: 0.6594
iteration 13/100 - Loss: 1.9775 - Accuracy: 0.6705
iteration 14/100 - Loss: 1.8897 - Accuracy: 0.6850
iteration 15/100 - Loss: 1.7933 - Accuracy: 0.6984
iteration 16/100 - Loss: 1.6933 - Accuracy: 0.7084
iteration 17/100 - Loss: 1.5950 - Accuracy: 0.7144
iteration 18/100 - Loss: 1.5026 - Accuracy: 0.7198
iteration 19/100 - Loss: 1.4188 - Accuracy: 0.7249
iteration 20/100 - Loss: 1.3442 - Accura

Results on test data

In [13]:
dense1.forward(X_test)
ReLU.forward(dense1.output)
dense2.forward(ReLU.output)
softmax.forward(dense2.output)
test_acc = accuracy(softmax.output, y_test)
print(f"Test Accuracy: {test_acc:.4f}")

Test Accuracy: 0.8652
